# Multi Layer Perceptron

In [8]:
import sys

sys.path.append("..")
from pprint import pprint

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

from models.mlp import MultiLayerPerceptron
from models.activation_functions import ReLu
from models.loss_functions import LogLoss
from models.weight_initializer import NormalInitializer

In [9]:
heart_disease = fetch_ucirepo(id=45)

X_raw: pd.DataFrame = heart_disease.data.features
y_raw: pd.Series = (heart_disease.data.targets["num"] > 0).astype(int)

In [10]:
mask = ~np.asarray(pd.isna(X_raw).any(axis=1))

X = X_raw[mask]
y = y_raw[mask]

In [11]:
np.asarray(X[:1]).shape

(1, 13)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUM_COLS = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

prep = ColumnTransformer(
    transformers=[
        ("stand", StandardScaler(), NUM_COLS),
    ],
    remainder="passthrough",
)

In [13]:
loss_function = LogLoss()

In [14]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y.to_numpy(), test_size=0.2, stratify=y, random_state=42
)

hidden_layer_architectures = [
    [13],
    [67],
    [8],
    [13, 13, 13],
    [32, 16, 8, 4, 2],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [2, 4, 8, 16, 32],
    [24, 16],
    [8, 24],
]

results = []

for hidden_layer_sizes in hidden_layer_architectures:
    for use_standard_scaler in [True, False]:
        # Reset the seed so both preprocessing variants use the same initial weights.
        np.random.seed(42)

        pipe = Pipeline(
            [
                ("prep", clone(prep) if use_standard_scaler else "passthrough"),
                (
                    "clf",
                    MultiLayerPerceptron(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation_function=ReLu(),
                        loss_function=loss_function,
                        weight_initializer=NormalInitializer(0.1),
                        learning_rate=0.01,
                        max_iter=1000,
                    ),
                ),
            ]
        )

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        results.append(
            {
                "hidden_layer_sizes": str(hidden_layer_sizes),
                "standard_scaler": use_standard_scaler,
                "accuracy": accuracy_score(y_test, y_pred),
                "f1": f1_score(y_test, y_pred, zero_division=0),
                "precision": precision_score(y_test, y_pred, zero_division=0),
                "recall": recall_score(y_test, y_pred, zero_division=0),
            }
        )

        pprint(results[-1])

results_df = pd.DataFrame(results)
results_df.sort_values(
    by=["hidden_layer_sizes", "standard_scaler"],
    ascending=[True, False],
).reset_index(drop=True)

{'accuracy': 0.85,
 'f1': 0.8235294117647058,
 'hidden_layer_sizes': '[13]',
 'precision': 0.9130434782608695,
 'recall': 0.75,
 'standard_scaler': True}
{'accuracy': 0.75,
 'f1': 0.7058823529411765,
 'hidden_layer_sizes': '[13]',
 'precision': 0.782608695652174,
 'recall': 0.6428571428571429,
 'standard_scaler': False}
{'accuracy': 0.8166666666666667,
 'f1': 0.7924528301886793,
 'hidden_layer_sizes': '[67]',
 'precision': 0.84,
 'recall': 0.75,
 'standard_scaler': True}
{'accuracy': 0.5333333333333333,
 'f1': 0.0,
 'hidden_layer_sizes': '[67]',
 'precision': 0.0,
 'recall': 0.0,
 'standard_scaler': False}
{'accuracy': 0.8833333333333333,
 'f1': 0.8679245283018868,
 'hidden_layer_sizes': '[8]',
 'precision': 0.92,
 'recall': 0.8214285714285714,
 'standard_scaler': True}
{'accuracy': 0.5333333333333333,
 'f1': 0.0,
 'hidden_layer_sizes': '[8]',
 'precision': 0.0,
 'recall': 0.0,
 'standard_scaler': False}
{'accuracy': 0.8333333333333334,
 'f1': 0.8214285714285714,
 'hidden_layer_sizes':

,hidden_layer_sizes,standard_scaler,accuracy,f1,precision,recall
0,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",True,0.533333,0.000000,0.000000,0.000000
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",False,0.533333,0.000000,0.000000,0.000000
2,"[13, 13, 13]",True,0.833333,0.821429,0.821429,0.821429
3,"[13, 13, 13]",False,0.716667,0.760563,0.627907,0.964286
4,[13],True,0.850000,0.823529,0.913043,0.750000
5,[13],False,0.750000,0.705882,0.782609,0.642857
6,"[2, 4, 8, 16, 32]",True,0.533333,0.000000,0.000000,0.000000
7,"[2, 4, 8, 16, 32]",False,0.766667,0.730769,0.791667,0.678571
8,"[24, 16]",True,0.883333,0.862745,0.956522,0.785714
9,"[24, 16]",False,0.700000,0.550000,0.916667,0.392857
